# 01 — Pull ACLED Events

**Source:** Armed Conflict Location & Event Data (ACLED)  
**Access:** REST API — free registration at [acleddata.com](https://acleddata.com)  
**Coverage:** Global, 1997–present, event-level  
**Frequency:** Real-time (this notebook pulls a configurable date range)  

## What this notebook does
1. Pages through the ACLED API to pull all global events within `START_DATE`–`END_DATE`
2. Writes the raw event-level parquet to ADLS: `raw/acled/events/`
3. Aggregates to **country-month** counts (event types + fatalities) used as predictor features
4. Writes the monthly aggregate parquet to ADLS: `raw/acled/monthly_agg/`

## Required environment variables
```
ACLED_API_KEY      — API key from acleddata.com
ACLED_EMAIL        — Email registered with ACLED
ADLS_ACCOUNT_NAME  — Azure storage account name
ADLS_CONTAINER     — Container name (e.g. 'data')
```

## Key fields pulled
All 31 fields from the ACLED 2024 schema (see DatasetVariableSynthesis.md §1.1).
The monthly aggregate produces the predictor features used in Domain 4 of the meta-plan.

In [ ]:
import os
import time
import requests
import pandas as pd
from datetime import datetime
from tqdm.notebook import tqdm
from azure.identity import DefaultAzureCredential

## Configuration
Edit the variables below or set them as environment variables before running.

In [ ]:
# ── Load ACLED credentials from Key Vault ──────────────────────────────────
# KEY_VAULT_URL must be set as a compute instance environment variable.
# Populates ACLED_API_KEY and ACLED_EMAIL in os.environ before the config cell reads them.
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve().parent.parent))  # repo root
from utils.keyvault import load_acled_credentials
load_acled_credentials()

In [ ]:
# --- ACLED credentials ---
ACLED_API_KEY = os.environ["ACLED_API_KEY"]
ACLED_EMAIL   = os.environ["ACLED_EMAIL"]

# --- Date range (inclusive) ---
START_DATE = "2000-01-01"
END_DATE   = datetime.utcnow().strftime("%Y-%m-%d")

# --- ADLS target ---
ADLS_ACCOUNT_NAME = os.environ["ADLS_ACCOUNT_NAME"]
ADLS_CONTAINER    = os.getenv("ADLS_CONTAINER", "data")

# --- API settings ---
ACLED_BASE_URL = "https://api.acleddata.com/acled/read"
PAGE_SIZE      = 5000   # maximum allowed by ACLED
RETRY_WAIT_S   = 5      # seconds between retries on 429/5xx
MAX_RETRIES    = 3

RUN_DATE = datetime.utcnow().strftime("%Y%m%d")
print(f"Pull window : {START_DATE} → {END_DATE}")
print(f"ADLS target : abfss://{ADLS_CONTAINER}@{ADLS_ACCOUNT_NAME}.dfs.core.windows.net/raw/acled/")
print(f"Run date    : {RUN_DATE}")

## ADLS helper

In [ ]:
credential = DefaultAzureCredential()

storage_options = {
    "account_name": ADLS_ACCOUNT_NAME,
    "credential": credential,
}

def adls_path(subpath: str) -> str:
    return (
        f"abfss://{ADLS_CONTAINER}@{ADLS_ACCOUNT_NAME}"
        f".dfs.core.windows.net/{subpath}"
    )

def write_parquet(df: pd.DataFrame, subpath: str) -> None:
    path = adls_path(subpath)
    df.to_parquet(path, storage_options=storage_options, index=False, engine="pyarrow")
    print(f"  Written {len(df):,} rows → {path}")

## ACLED pagination helper

ACLED caps each response at 5,000 records. We page until the API returns fewer
records than `PAGE_SIZE`, which signals the last page.

In [ ]:
def fetch_acled_page(page: int) -> list[dict]:
    """Fetch one page of ACLED events; returns list of event dicts."""
    params = {
        "key":              ACLED_API_KEY,
        "email":            ACLED_EMAIL,
        "event_date":       f"{START_DATE}|{END_DATE}",
        "event_date_where": "BETWEEN",
        "limit":            PAGE_SIZE,
        "page":             page,
    }
    for attempt in range(1, MAX_RETRIES + 1):
        resp = requests.get(ACLED_BASE_URL, params=params, timeout=60)
        if resp.status_code == 200:
            return resp.json().get("data", [])
        if resp.status_code in (429, 500, 502, 503):
            print(f"  HTTP {resp.status_code} on page {page}, attempt {attempt}/{MAX_RETRIES} — waiting {RETRY_WAIT_S}s")
            time.sleep(RETRY_WAIT_S * attempt)
        else:
            resp.raise_for_status()
    raise RuntimeError(f"Failed to fetch page {page} after {MAX_RETRIES} retries")


def pull_all_acled() -> pd.DataFrame:
    """Pull all ACLED events in the configured date range via pagination."""
    all_records: list[dict] = []
    page = 1
    pbar = tqdm(desc="ACLED pages", unit="page")
    while True:
        records = fetch_acled_page(page)
        all_records.extend(records)
        pbar.update(1)
        pbar.set_postfix({"total_events": len(all_records)})
        if len(records) < PAGE_SIZE:
            break  # last page
        page += 1
    pbar.close()
    return pd.DataFrame(all_records)

## Pull raw events

In [ ]:
print("Pulling ACLED events...")
df_raw = pull_all_acled()
print(f"Total events pulled: {len(df_raw):,}")
df_raw.head(2)

## Type normalisation

Cast key columns to correct dtypes; ACLED returns everything as strings.

In [ ]:
int_cols   = ["year", "time_precision", "inter1", "inter2", "interaction",
              "iso", "geo_precision", "fatalities", "timestamp"]
float_cols = ["latitude", "longitude"]
date_cols  = ["event_date"]

for c in int_cols:
    if c in df_raw.columns:
        df_raw[c] = pd.to_numeric(df_raw[c], errors="coerce").astype("Int64")

for c in float_cols:
    if c in df_raw.columns:
        df_raw[c] = pd.to_numeric(df_raw[c], errors="coerce")

for c in date_cols:
    if c in df_raw.columns:
        df_raw[c] = pd.to_datetime(df_raw[c], errors="coerce")

# Derive year_month for aggregation joins
df_raw["year_month"] = df_raw["event_date"].dt.to_period("M").astype(str)

print(df_raw.dtypes)
print(f"\nDate range: {df_raw['event_date'].min()} → {df_raw['event_date'].max()}")
print(f"Countries : {df_raw['country'].nunique()}")

## Write raw events to ADLS

In [ ]:
write_parquet(df_raw, f"raw/acled/events/{RUN_DATE}/acled_events.parquet")

## Build country-month aggregates

These columns become the **Domain 4 — Event History** predictor features in the panel:

| Feature | Description |
|---|---|
| `events_total` | Total events per country-month |
| `events_battles` | Battles sub-count |
| `events_protests` | Protests sub-count |
| `events_riots` | Riots sub-count |
| `events_vac` | Violence against civilians sub-count |
| `events_explosions` | Explosions/remote violence sub-count |
| `fatalities_total` | Sum of fatalities |
| `civilian_targeting_count` | Events with civilian targeting flag |

In [ ]:
df_raw["is_battle"]    = (df_raw["event_type"] == "Battles").astype(int)
df_raw["is_protest"]   = (df_raw["event_type"] == "Protests").astype(int)
df_raw["is_riot"]      = (df_raw["event_type"] == "Riots").astype(int)
df_raw["is_vac"]       = (df_raw["event_type"] == "Violence against civilians").astype(int)
df_raw["is_explosion"] = (df_raw["event_type"] == "Explosions/Remote violence").astype(int)
df_raw["is_civ_target"]= (df_raw["civilian_targeting"].notna() &
                           (df_raw["civilian_targeting"] != "")).astype(int)

agg_map = {
    "event_id_cnty":  "count",
    "is_battle":      "sum",
    "is_protest":     "sum",
    "is_riot":        "sum",
    "is_vac":         "sum",
    "is_explosion":   "sum",
    "fatalities":     "sum",
    "is_civ_target":  "sum",
}

df_monthly = (
    df_raw
    .groupby(["iso", "country", "year_month"], as_index=False)
    .agg(agg_map)
    .rename(columns={
        "event_id_cnty": "events_total",
        "is_battle":     "events_battles",
        "is_protest":    "events_protests",
        "is_riot":       "events_riots",
        "is_vac":        "events_vac",
        "is_explosion":  "events_explosions",
        "fatalities":    "fatalities_total",
        "is_civ_target": "civilian_targeting_count",
    })
)

df_monthly["fatalities_total"] = df_monthly["fatalities_total"].astype("Int64")

print(f"Monthly aggregate shape: {df_monthly.shape}")
df_monthly.head()

## Write monthly aggregates to ADLS

In [ ]:
write_parquet(df_monthly, f"raw/acled/monthly_agg/{RUN_DATE}/acled_monthly.parquet")

## Summary

In [ ]:
print("=" * 55)
print("ACLED pull complete")
print("=" * 55)
print(f"  Raw events        : {len(df_raw):,} rows")
print(f"  Monthly aggregates: {len(df_monthly):,} country-months")
print(f"  Countries covered : {df_raw['country'].nunique()}")
print(f"  Date range        : {df_raw['event_date'].min().date()} → {df_raw['event_date'].max().date()}")
print()
print("ADLS paths written:")
print(f"  raw/acled/events/{RUN_DATE}/acled_events.parquet")
print(f"  raw/acled/monthly_agg/{RUN_DATE}/acled_monthly.parquet")